## Building a simple ReAct Agent from Scratch

In [1]:
# Loading the keys
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [2]:
import openai
import os
import httpx
import re


In [3]:
from openai import OpenAI
client = OpenAI()

model = 'gpt-4o-mini'
prompt = "Write something short but funny."

chat_completion = client.chat.completions.create(
    model=model,
    messages = [
        {
            'role': 'user',
            'content': prompt
        }
    ]
)


In [4]:
print(chat_completion.choices[0].message.content)

Why did the scarecrow win an award? Because he was outstanding in his field!


### Create Agent class

In [19]:
class Agent:
    def __init__(self, system=''):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({'role': 'system', 'content': self.system})

    def __call__(self, prompt):
        self.messages.append({'role': 'user', 'content': prompt})
        result = self.execute()
        self.messages.append({'role': 'assistant', 'content': result})
        return result

    def execute(self, model="gpt-4o-mini", temperature=0):
        completions = client.chat.completions.create(
            model=model,
            messages=self.messages,
            temperature=temperature
        )
        return completions.choices[0].message.content

### Creating a ReAct Prompt

In [7]:
prompt = '''
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_cost:
e.g. get_cost: book
returns the cost of a book

Example session #1:

Question: How much does a pen cost?
Thought: I should look the pen cost using get_cost
Action: get_cost: pen
PAUSE

You will be called again with this:

Observation: A pen costs $5

You then output:

Answer: A pen costs $5
'''.strip()


### Create Tools

In [10]:
def calculate(what):
    return eval(what)

def get_cost(thing):
    if thing in 'pen':
        return('A pen costs $5')
    elif thing in 'book':
        return('A book costs $20')
    elif thing in 'stapler':
        return('A stapler costs $10')
    else:
        return('A random thing for writing costs $12')

# TODO -> Getting forbidden for Wikipedia
# def wikipedia(query):
#     response = httpx.get('https://en.wikipedia.org/w/api.php', params = {
#         'action': 'query',
#         'list': 'search',
#         'srsearch': query,
#         'format': 'json'
#     })
#     print(response)
#     results = response.json().get('query').get('search', [])

#     if not results:
#         return None
#     return results[0]['snippet']

In [40]:
known_actions = {
    'calculate': calculate,
    'get_cost': get_cost
}

### Testing an Agent

In [23]:
agent = Agent(prompt)

In [24]:
result = agent("How much does a pen cost?")
print(result)

Thought: I should look up the cost of a pen using the get_cost action.  
Action: get_cost: pen  
PAUSE


In [25]:
print(agent.messages)

[{'role': 'system', 'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_cost:\ne.g. get_cost: book\nreturns the cost of a book\n\nExample session #1:\n\nQuestion: How much does a pen cost?\nThought: I should look the pen cost using get_cost\nAction: get_cost: pen\nPAUSE\n\nYou will be called again with this:\n\nObservation: A pen costs $5\n\nYou then output:\n\nAnswer: A pen costs $5'}, {'role': 'user', 'content': 'How much does a pen cost?'}, {'role': 'assistant', 'content': 'Thought: I should look up the cost of a pen using the

In [26]:
next_prompt = f"Observtion: {get_cost('pen')}"

In [27]:
agent(next_prompt)

'Answer: A pen costs $5.'

In [28]:
agent.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_cost:\ne.g. get_cost: book\nreturns the cost of a book\n\nExample session #1:\n\nQuestion: How much does a pen cost?\nThought: I should look the pen cost using get_cost\nAction: get_cost: pen\nPAUSE\n\nYou will be called again with this:\n\nObservation: A pen costs $5\n\nYou then output:\n\nAnswer: A pen costs $5'},
 {'role': 'user', 'content': 'How much does a pen cost?'},
 {'role': 'assistant',
  'content': 'Thought: I should look up the cost of a pen usi

#### Complicated question

In [33]:
new_agent = Agent(prompt)

In [34]:
question = "I want to buy a pen and a book. How much do they cost in total?"
new_agent(question)

'Thought: I need to find the cost of both a pen and a book to calculate the total cost. I will first get the cost of the pen and then the cost of the book. \n\nAction: get_cost: pen\nPAUSE'

In [35]:
next_prompt = f"Observation : {get_cost('pen')}"
new_agent(next_prompt)

'Thought: Now that I know the cost of the pen is $5, I need to find out the cost of the book to calculate the total cost. \n\nAction: get_cost: book\nPAUSE'

In [36]:
next_prompt = f"Observation: {get_cost('book')}"
new_agent(next_prompt)

'Thought: I have the costs now: the pen costs $5 and the book costs $20. I can add these two amounts together to find the total cost. \n\nAction: calculate: 5 + 20\nPAUSE'

In [38]:
next_prompt = f"Observation: {calculate("5+10")}"
new_agent(next_prompt)

'Answer: The total cost of a pen and a book is $25.'

NOTE:   
In the above example everything was done manually. We can automate this loop so LLM takes over and calls all the functions and return us the output when it reaches the answer.

### Automating the Agent

In [39]:
# define a regex for finding the action string
action_re = re.compile(r'^Action: (\w+): (.*)$') # python regular expression to select Action.

In [41]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question

    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)

        # using regex to parse the response from the agent.
        actions = [ # This is a list comprehension
            action_re.match(a) for a in result.split('\n') if action_re.match(a)
        ]

        if actions:
            action, action_input = actions[0].groups()
            
            if action not in known_actions:
                raise Exception(f'Unknown action: {action}: {action_input}')

            print(f' -- running {action} {action_input}')
            observation = known_actions[action](action_input)

            print(f'Observation: {observation}')
            next_prompt = f'Observation {observation}'
        else:
            return

In [42]:
question = '''I want to buy 2 books and 3 pens. How much do I have to pay?'''
query(question)

Thought: I need to find the cost of both the books and the pens to calculate the total amount. First, I will get the cost of a book and then the cost of a pen. After that, I can calculate the total cost based on the quantities. 

Action: get_cost: book
PAUSE
 -- running get_cost book
Observation: A book costs $20
Thought: Now that I know the cost of a book is $20, I need to find out the cost of a pen to complete the calculation for the total cost. 

Action: get_cost: pen
PAUSE
 -- running get_cost pen
Observation: A pen costs $5
Thought: I have the costs now: a book costs $20 and a pen costs $5. I need to calculate the total cost for 2 books and 3 pens. The total cost can be calculated as (2 * cost of book) + (3 * cost of pen).

Action: calculate: (2 * 20) + (3 * 5)
PAUSE
 -- running calculate (2 * 20) + (3 * 5)
Observation: 55
Answer: You have to pay $55 for 2 books and 3 pens.
